In [ ]:
# 1. Install dependencies
!pip install -q google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client anthropic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 6.4 MB/s eta 0:00:00


## 2. Upload your Gmail OAuth credentials file

In [ ]:
from google.colab import files

print("Upload the credentials.json you downloaded from Google Cloud Console:")
uploaded = files.upload()
CREDENTIALS_PATH = list(uploaded.keys())[0]
print(f"Using credentials file: {CREDENTIALS_PATH}")


Upload the credentials.json you downloaded from Google Cloud Console:


## 3. Authenticate with Gmail

Colab can't open a local browser redirect, so this uses the manual
copy-paste OAuth flow: run the cell, open the printed link, approve
access, copy the code Google gives you, and paste it back below.


In [ ]:
from google_auth_oauthlib.flow import Flow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
import json, os

GMAIL_SCOPES = ["https://www.googleapis.com/auth/gmail.modify"]

flow = Flow.from_client_secrets_file(
    CREDENTIALS_PATH,
    scopes=GMAIL_SCOPES,
    redirect_uri="urn:ietf:wg:oauth:2.0:oob",
)

auth_url, _ = flow.authorization_url(prompt="consent")
print("1. Open this URL in a browser:\n")
print(auth_url)
print("\n2. Approve access, copy the code shown, and paste it below.")

auth_code = input("Enter the authorization code: ").strip()
flow.fetch_token(code=auth_code)
creds = flow.credentials

gmail_service = build("gmail", "v1", credentials=creds)
my_email = gmail_service.users().getProfile(userId="me").execute()["emailAddress"]
print(f"\nAuthenticated as: {my_email}")


## 4. Set your Anthropic API key

In [ ]:
from getpass import getpass
import anthropic

ANTHROPIC_API_KEY = getpass("Paste your Anthropic API key: ")
claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
CLAUDE_MODEL = "claude-sonnet-4-6"
print("Claude client ready.")


## 5. Configure safety rules

Same idea as the script version:
- `"draft_only"` — never auto-sends, just creates Gmail drafts for you to review. **Start here.**
- `"whitelist"` — auto-sends only to senders/domains in `AUTO_SEND_WHITELIST`.
- `"all"` — auto-sends every reply. Use only once you trust the output.


In [ ]:
AUTO_SEND_MODE = "draft_only"   # "draft_only" | "whitelist" | "all"

AUTO_SEND_WHITELIST = [
    # "boss@company.com",
    # "@kkwagh.edu.in",
]

IGNORE_SENDERS = [
    "noreply@", "no-reply@", "notifications@", "mailer-daemon@",
]

ONLY_PROCESS_NEW_MAIL = True   # skip mail that arrived before this session started
MAX_EMAILS_PER_CYCLE = 10
POLL_INTERVAL_SECONDS = 60

SYSTEM_PROMPT = """You are drafting email replies on behalf of Kunal, a B.Tech AI & Data Science
Engineering student at K.K. Wagh Institute of Engineering, Nashik.

Guidelines:
- Keep replies concise, polite, and professional (or casual if the incoming email is casual).
- Match the tone and formality of the sender's email.
- Do not invent facts, commitments, dates, or promises not present in the thread.
- If the email requires information you don't have (e.g. specific data, decisions,
  approvals), draft a reply that asks a clarifying question instead of guessing.
- Sign off as \"Kunal\".
- Output ONLY the email body text. No subject line, no explanation, no markdown.
"""

print(f"Mode: {AUTO_SEND_MODE}")


## 6. Helper functions (Gmail parsing, drafting, sending)

In [ ]:
import base64, re, time
from email.mime.text import MIMEText
from email.utils import parseaddr


def list_unread_messages(service):
    results = service.users().messages().list(
        userId="me", labelIds=["UNREAD", "INBOX"], maxResults=MAX_EMAILS_PER_CYCLE
    ).execute()
    return results.get("messages", [])


def get_message(service, msg_id):
    return service.users().messages().get(userId="me", id=msg_id, format="full").execute()


def _get_header(message, name):
    for h in message["payload"].get("headers", []):
        if h["name"].lower() == name.lower():
            return h["value"]
    return ""


def _extract_body(payload):
    if payload.get("mimeType") == "text/plain" and "data" in payload.get("body", {}):
        return base64.urlsafe_b64decode(payload["body"]["data"]).decode("utf-8", errors="ignore")
    for part in payload.get("parts", []):
        text = _extract_body(part)
        if text:
            return text
    if payload.get("mimeType") == "text/html" and "data" in payload.get("body", {}):
        html = base64.urlsafe_b64decode(payload["body"]["data"]).decode("utf-8", errors="ignore")
        return re.sub("<[^<]+?>", "", html)
    return ""


def parse_message(message):
    return {
        "id": message["id"],
        "thread_id": message["threadId"],
        "from": _get_header(message, "From"),
        "subject": _get_header(message, "Subject"),
        "message_id_header": _get_header(message, "Message-ID"),
        "body": _extract_body(message["payload"]).strip(),
    }


def should_ignore(sender_email):
    sender_email = sender_email.lower()
    return any(p in sender_email for p in IGNORE_SENDERS)


def is_whitelisted(sender_email):
    sender_email = sender_email.lower()
    return any(e.lower() in sender_email for e in AUTO_SEND_WHITELIST)


def _build_reply(parsed, reply_body, my_email):
    _, sender_email = parseaddr(parsed["from"])
    subject = parsed["subject"]
    if not subject.lower().startswith("re:"):
        subject = f"Re: {subject}"
    mime_msg = MIMEText(reply_body)
    mime_msg["to"] = sender_email
    mime_msg["from"] = my_email
    mime_msg["subject"] = subject
    if parsed["message_id_header"]:
        mime_msg["In-Reply-To"] = parsed["message_id_header"]
        mime_msg["References"] = parsed["message_id_header"]
    raw = base64.urlsafe_b64encode(mime_msg.as_bytes()).decode()
    return {"raw": raw, "threadId": parsed["thread_id"]}


def create_draft_reply(service, parsed, reply_body, my_email):
    body = _build_reply(parsed, reply_body, my_email)
    return service.users().drafts().create(userId="me", body={"message": body}).execute()


def send_reply(service, parsed, reply_body, my_email):
    body = _build_reply(parsed, reply_body, my_email)
    return service.users().messages().send(userId="me", body=body).execute()


def mark_as_read(service, msg_id):
    service.users().messages().modify(
        userId="me", id=msg_id, body={"removeLabelIds": ["UNREAD"]}
    ).execute()


def draft_reply_with_claude(parsed_email):
    user_prompt = (
        f"From: {parsed_email['from']}\n"
        f"Subject: {parsed_email['subject']}\n\n"
        f"Email body:\n{parsed_email['body']}\n\n"
        "Write the reply body."
    )
    response = claude_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=600,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}],
    )
    return "".join(b.text for b in response.content if b.type == "text").strip()


def decide_action(sender_email):
    if AUTO_SEND_MODE == "all":
        return "send"
    if AUTO_SEND_MODE == "whitelist":
        return "send" if is_whitelisted(sender_email) else "draft"
    return "draft"

print("Helper functions loaded.")


## 7. Run one poll cycle

Run this cell whenever you want to check for new mail and process it.
Re-run it manually, or use the looping cell in Step 8 for repeated polling.


In [ ]:
def process_once():
    messages = list_unread_messages(gmail_service)
    if not messages:
        print("No new unread messages.")
        return

    for m in messages:
        full = get_message(gmail_service, m["id"])
        parsed = parse_message(full)
        _, sender_email = parseaddr(parsed["from"])

        if not sender_email or sender_email.lower() == my_email.lower():
            continue
        if should_ignore(sender_email):
            print(f"Skipping ignored sender: {sender_email}")
            continue
        if ONLY_PROCESS_NEW_MAIL and int(full.get("internalDate", 0)) < SESSION_START_MS:
            continue

        print(f"\nNew mail from {sender_email} — Subject: {parsed['subject']}")

        try:
            reply_body = draft_reply_with_claude(parsed)
        except Exception as e:
            print(f"  Failed to draft reply: {e}")
            continue

        action = decide_action(sender_email)
        try:
            if action == "send":
                send_reply(gmail_service, parsed, reply_body, my_email)
                print(f"  -> Auto-sent reply to {sender_email}")
            else:
                create_draft_reply(gmail_service, parsed, reply_body, my_email)
                print("  -> Saved as draft for review")
            mark_as_read(gmail_service, m["id"])
        except Exception as e:
            print(f"  Failed to {action} reply: {e}")


SESSION_START_MS = int(time.time() * 1000)
process_once()


## 8. (Optional) Keep polling in a loop

Runs until you stop the cell (Colab: the ■ stop button) or the runtime
disconnects. Remember Colab has an inactivity/session time limit, so
this is best for short monitoring sessions, not a permanent deployment.


In [ ]:
try:
    while True:
        process_once()
        time.sleep(POLL_INTERVAL_SECONDS)
except KeyboardInterrupt:
    print("Stopped.")
